# Identifier les interventions mobilisant l'idée de "République"

Ce notebook/guideline vise à présenter le stratégie de recherche pour circonscrire l'analyse de discours aux prises de paroles politiques mobilisant l'idée de "République". 
(Différence par rapport au notebook final : pré-traitement de texte effectué à ce moment-là et pas ici + distinction en étapes pour AT)

## Définition préalable de ce concept : 

Énoncés sur la "République" captés par le concept "d'idée politique" au sens d'effectuer un travail sur un ensemble idéel de références organisées à la "République". 

***Nécessité d'écrire une véritable définition !!!!!***

    ==> Statut dual de l'idée de "République" :
        - (1) Mobilisations de références directes à la "République" via l'utilisation de la famille du mot "République" (« république », "républicain", etc.),
        - (2) Mobilisation de références indirectes et implicites à la "République" via son champ lexical. Peut alors comprendre des figures, évènements et valeurs/principes dont la définition/liste n'est pas consensuelle dans la littérature.

## Stratégie de recherche : 

1. Définition d'une regex large inclusive sur la famille de mot "république" [V]
2. Entrainer un classifieur pour exclure les termes de la famille de mot "république" ne renvoyant pas à cette idée
3. Comparer le classifieur à une regex exclusive
    - Améliorer la regex exclusive et la simplifier 
4. Définition d'une regex large pour comprendre tout le champ lexical possible de la "république"
5. Trier les références à l'IR des références extérieures ou indépendantes de l'IR
    - Vérifier le nombre de co-occurences entre "République" (1), les éventuelles références (2), et les co-occurrences entre valeurs et figures, ou valeurs et évènements pour mettre l'accent sur ce qu'il faudrait potentiellement exclure de l'idée de République = essayer d’objectiver si il existe un champ lexical +/- cohérent qui se détache (et ce qui serait étranger/extérieur)  
    - Exclure par annotation manuelle (sur active tigger) les valeurs, figures et évènements qui en fonction de l'auteur, de la date et du contexte linguistique renvoie à une autre idée ou idéologie politique celle de la République

In [28]:
# Ouvrir le fichier propre 
import pandas as pd
import re
import csv

df = pd.read_csv(
    "../data/interim/Data_AN_CSS_clean.csv", low_memory=False, dtype={"ID_orateur": str}
)
df.shape

FileNotFoundError: [Errno 2] No such file or directory: '../data/interim/Data_AN_CSS_clean.csv'

### Étape 1 : Regex large famille du mot "République"

Comment définir largement la famille du mot "République" ? En revenant à sa "racine" puisque la famille d'un mot correspond à l'ensemble des mots dérivés de la même racine (ou base/radical) ou d'une racine proche (mer et mar par exemple). Ces mots prennent des formes différentes (verbe, adverbe, nom, adjectif, etc.) en changeant de suffixe, pré-fixe, etc.

Définition théorique : "*on donne le nom de « racine » à cette partie du lexème qui constitue la limite de la segmentation, à la fois porteuse de l’identité du lexème (cette partie d’interprétation qui le différencie de tous les autres lexèmes), et qui est insécable sous peine que soit perdue cette identité lexicale.*" Huot, H. (2006). Chapitre III. Racines et radicaux. Cursus, 2, 41-52.

Ici la base de "République" est "républi" : exemple (républicain, républicanisme, )

In [ ]:
# Regex de la famille du mot "République" (simplifié ici) --> passe de 683680 à 37979.

pattern_lexical = re.compile(
    r"républi",
    re.I,
)

def famille_de_mot(texte_propre):
    for match in pattern_lexical.finditer(texte_propre):
        return True
        return False

In [ ]:
# Appliquer sur la colonne
df["texte_propre"] = df["texte_propre"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["texte_propre"] = df["texte_propre"].apply(famille_de_mot)

In [ ]:
df_répu_in = df[df["République_FDM_IN"] == True]

In [ ]:
df_répu_in.shape

(37979, 27)

In [ ]:
import csv  

df_répu_in.to_csv(
    "/Users/matthiaslevalet/Desktop/Projet de recherche/CSS_République/Data/Interim/Data_AN_CSS_République_inclusive.csv",
    index=False,
    quoting=csv.QUOTE_ALL,  # a permis de résoudre le soucis d'écart. Checker
)

### Étape 2/3 : Constitution d'une regex exclusive 

De nombres termes appartiennent à la famille du mot "république" sans renvoyer à l'idée de "République" telle que définit ci-dessus, comme :

        * Représentant : “Président de la République” (pour dire le nom de la personne qui l’occupe)
        * Institutions :  “Cour de justice de la république”/"Cour de sûreté de la République"/"procureur de la République"/"Administration générale de la république"
        * Partis ou groupes politiques : "L(l)es Républicains"/"La République en Marche"/"groupe socialistes, écologistes et républicains" (Socialiste, Écologiste et Républicain)/"Gauche démocrate et républicaine"
        * Pays (liste extensive créée pour une regex spécifique)

==> Écrire une regex pour les exclure automatiquement. 

Attention : comme les termes exclus peuvent apparaitre aussi avec les termes voulu, éviter de chainer et finir par virer des trucs qu'on aurait voulu (ex : "les idées républicaines sont menacées par Les Républicains")

In [ ]:
# Regex pays insuffisante car doit aussi comprendre la forme adjectivable des pays et république en minuscule 

# préparer les pays à exclure en utilisant la liste faite par le notebook pays_republique.ipynb
with open("/Users/matthiaslevalet/Desktop/Projet de recherche/CSS_République/Data/Interim/liste_pays_republique_sup.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays

# ici pas besoin d'avoir un groupe de capture pas pays mais juste global ok
# pattern_pays = r"(\b(?:" + r"|".join(re.escape(p) for p in liste_pays) + r")\b)"
pattern_pays = r"(?<!\w)(?:" + r"|".join(re.escape(p) for p in liste_pays) + r")(?!\w)" #fonction sans le \b car le b ne reconnait pas la bonne frontière de mot et laisse par exemple République d'Arménie

# = résultat en enlève 5 540 (de 37 979 regex inclusion, à 32 439 regex exclu uniquement pays_compile)


In [ ]:
# Regex des expressions à exclure

# Expressions à exclure - casse exacte 
pattern_excl_case_sensitive = re.compile(
    r"\b[LlDd]es Républicains\b" # (5 378 occurrences = 37 979 - 32 601 ? 18 555 = 32601 - 14046)
)  

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    r"|(\brépublique en marche\b)" #(3 547 occurrences = 17593 - 140046)
    r"|(\bprésident[s]? de la République\b)" #(13 377 occurences = 27 423- 14 046) 
    r"|(\bgauche démocrate et républicaine\b)" # (921 occurrences = 14 967 - 14046)
    r"|(\bsocialiste, écologiste et républicain\b)" # (1 occurrence = 14 065 - 14 046)
    r"|(\bprocureur[s]? de la République\b)" # (844 occurrences = 14890 - 14 046)
    r"|(\bcour[s]? de justice de la République\b)" # (89 occurrences= 14 135 - 14 046)
    r"|(\bcour[s]? de sûreté de la République\b)" # (11 occurrences = 14 057 - 14 046)
    r"|(\badministration générale de la République\b)" # (73 occurrences = 14119 - 14 046)
    r"|(" + pattern_pays + ")"  # ajout des exclusions de pays 5602 occurrences 5540 (sans le problème des 2 types d'apostrophes)
    r"|(\bGouvernement de la République française\b)" # 47 occurrences mais Pose question de si on met ou pas car surtout mobilisé lors de discussion sur la signature d'accords commerciaux entre deux pays (=signifie le gouvernement français + que la république comme idée politique)
    r"|(\badministration générale de la République\b)" #fait référence au nom d'une commission "commission des lois constitutionnelles, de la législation et de l’administration générale de la République"
    r"|(\brépublique[s] soviétique[s]\b)", # 6 occurrences 
    re.I,
)


def contains_lexical_outside_excl(text):
    # Trouver les positions des expressions exclues
    excl_positions = []

    # Ajouter les exclusions sensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )

    # Ajouter les exclusions insensibles à la casse
    excl_positions.extend(
         [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
     )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurences de la famille du mot 
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False

In [ ]:
# # bloc d'essai
mon_texte = "République arménienne"
contains_lexical_outside_excl(mon_texte)


False

In [ ]:
# Appliquer sur la colonne
df["texte_propre"] = df["texte_propre"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["Republique_FDM_Regex"] = df["texte_propre"].apply(contains_lexical_outside_excl)

In [ ]:
df_match = df[df["Republique_FDM_Regex"]]
df_match.shape

(13931, 26)

In [ ]:
df

(683680, 26)

On passe de 37979 (regex inclusion) à 13931 (regex exclusion) (à comparer avec le clasifieur entrainé sur AT)

In [ ]:
df_match.to_csv(
    "/Users/matthiaslevalet/Desktop/Projet de recherche/CSS_République/Data/Interim/Data_AN_CSS_République_FDM.csv",
    index=False,
    quoting=csv.QUOTE_ALL,  # a permis de résoudre le soucis d'écart. Checker
)

In [ ]:
# vérif écriture/lecture ok
df_test = pd.read_csv(".../Republique_FDM_Regex.csv")
df_test.shape

FileNotFoundError: [Errno 2] No such file or directory: '.../Republique_FDM_Regex.csv'

# PAUSE ICI

## Étape 3 : Regex inclusive large du champ lexical de la "République"

### Dictionnaire manuel des familles de mots pouvant appartenir au champ lexical de la "République" 

À compléter


Bibliographie indicative : 

Christin, O.,  Soulié, S.  et Worms, F.  (2023). Les 100 mots de la République. (2e éd.). Presses Universitaires de France. https://shs.cairn.info/les-100-mots-de-la-republique--9782715414075?lang=fr.

Duclert, V., & Prochasson, C. (2002). Dictionnaire critique de la République. Flammarion.

Spitz, J.-F. (2022). La République ? Quelles valeurs ? Essai sur un nouvel intégrisme politique. Gallimard; Cairn.info. https://doi.org/10.3917/gall.spitz.2022.01


#### Valeurs/Principes

In [2]:
import pandas as pd
import re
import csv

df_regroup = pd.read_csv(
    "../data/interim/df_regroup_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)
df_regroup.shape

(425545, 54)

In [15]:
import pandas as pd
import re
import csv

df = pd.read_csv(
    "../data/interim/df_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)
df.shape

(683660, 54)

In [41]:
# Regex des familles des mots pouvant correspondre à des "valeurs de la république"

pattern_valeurs = re.compile(
    r"(libert\w*)"                              # Liberté, libéral, libération...
    r"|(\b[ée]gal\w*)"                          # Égalité, égalitaire...
    r"|(\bfratern\w*)"                          # Fraternité, fraternel...
    r"|(\bindivis\w*)"                          # Indivisible, indivisibilité...
    r"|(\bla[ïi]c\w*)"                          # Laïc, laïcité...
    r"|(\bd[ée]mocrat\w*)"                      # Démocratie, démocratique...
    r"|(\bsoci\w*)"                             # Social, société...
    r"|(\bsouverain\w*)"                        # Souverain, souveraineté...
    r"|(\bunit\w*)"                             # Unité, unification...
    r"|(\bunivers\w*)"                          # Universel, universalisme...
    r"|(\bindiff[ée]r\w*)"                      # Indifférence, indifférent...
    r"|(\bdroit\w*\s*(de\s*l['’]?\s*)?homme\w*)" # Droits de l’homme...
    r"|(\bcitoyen\w*)"                          # Citoyen, citoyenneté...
    r"|(\bddhc\b)"                              # DDHC (Déclaration des Droits de l’Homme et du Citoyen)
    r"|(\bgouvern\w*\s*(du|par|pour)\s+peuple\w*)"  # Gouvernement du peuple...
    r"|(\bchance\w*)"                           # Égalité des chances
    ,
    re.I,
)

def famille_de_mot_valeurs(texte_propre: str) -> bool:
    return bool(pattern_valeurs.search(texte_propre))

In [42]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_Valeurs"] = df["texte"].apply(famille_de_mot_valeurs)

In [43]:
df

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,dateMaj,dateSeance_ts,groupe_députés_affiliation,groupe&gvt_affiliation,groupe_all_affiliation,Texte_clean,repu_match_valide,FDM_Valeurs,FDM_Figures,FDM_dates
0,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,2025-09-26,2018-06-02 09:30:00,LR,LR,LR,L'article 25 concerne les organismes HLM et le...,False,True,False,False
1,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,2025-09-26,2018-06-02 09:30:00,DEM,DEM,DEM,"Pour cette intervention sur l'article, je remp...",True,True,False,False
2,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,2025-09-26,2018-06-02 09:30:00,LR,LR,LR,C'est une réalité !,False,False,False,False
3,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,2025-09-26,2018-06-02 09:30:00,NI,NI,NI,C'est vrai !,False,False,False,False
4,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,2025-09-26,2018-06-02 09:30:00,DEM,DEM,DEM,"Nous devons tous en être conscients alors que,...",False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683655,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,2025-09-26,2023-02-27 16:00:00,NaN,GVT,REN,Il n'est pas un jour où je ne parle pas des se...,False,True,False,False
683656,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,2025-09-26,2023-02-27 16:00:00,GDR,GDR,GDR,"Monsieur le ministre, il faudrait des heures, ...",False,True,False,False
683657,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,2025-09-26,2023-02-27 16:00:00,NaN,GVT,REN,"Premièrement, est-il légitime pour l'État de s...",False,True,False,False
683658,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,2025-09-26,2023-02-27 16:00:00,SOC-A,SOC-A,SOC-A,Je remercie le groupe GDR-NUPES d'avoir mis ce...,False,False,False,False


In [ ]:
pattern_figures_sensitive = re.compile(
    r"(\bhussards\s+noirs\b)"
    r"|(\bles\s+lumières\b)"
    r"|(\bVictor\s+Hugo\b)"
    r"|(\bde\s+Gaulle\b)"
    r"|(\bde\s+Gouge\b)"
    r"|(\bAlbert\s+l[’']ouvrier\b)"
    r"|(\bAlexandre\s+Martin\b)"
    r"|(\bEdgar\s+Faure\b)"
    r"|(\bMarianne\b)"
    r"|(\bRobespierre\b)"          
    r"|(\bClémenceau\b)"
    r"|(\bGambetta\b)"              
    r"|(\bVoltaire\b)"
    r"|(\bMontesquieu\b)"
    r"|(\bRousseau\b)"
    r"|(\bJean\s+Moulin\b)"
    r"|(\bles\s+Maquisards\b)"
    r"|(\bZola\b)"
    r"|(\bFerry\b)"
    r"|(\bCondorcet\b)"
    r"|(\bCarnot\b)"
    r"|(\bDelacroix\b)"
    r"|(\bJaur[eè]s\b)"             
    r"|(\bP[eé]guy\b)"              
    r"|(\bBaker\b)"
    r"|(\bRenouvier\b)"
    r"|(\bVeil\b)"
    r"|(\bBlum\b)"
    r"|(\bBriand\b)"
    r"|(\bBrunschvicg\b)"
    r"|(\bCassin\b)"
    r"|(\bCombes\b)"
    r"|(\bDebr[eé]\b)"
    r"|(\bDreyfus\b)"
    r"|(\bHerriot\b)"
    r"|(\bMonnet\b)"
    r"|(\bBadinter\b)"
    r"|(\bLavisse\b)"
    r"|(\bLedru[-\s]?Rollin\b)"     
    r"|(\bMandel\b)"
    r"|(\bMend[eè]s[-\s]?France\b)" 
    r"|(\bProudhon\b)"
    r"|(\bRaspail\b)"
    r"|(\bThiers\b)"
    r"|(\bWaldeck[-\s]?Rousseau\b)"
    r"|(\bJean\s+Zay\b)"
    r"|(\bDumas\b)"
    r"|(\bF[eé]lix\s+[ÉE]bou[eé]\b)" 
    r"|(\bLangevin\b)"
    r"|(\bPainlev[eé]\b)"
    r"|(\bBerthelot\b)"
    r"|(\bMarat\b)"
    r"|(\bPierre\s+Larousse\b)",
    re.I,
)

def famille_de_mot_figures(texte_propre: str) -> bool:
    return bool(pattern_figures_sensitive.search(texte_propre))


In [37]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_Figures"] = df["texte"].apply(famille_de_mot_figures)

In [38]:
df

,UID,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,scoreMajorite,dateMaj,dateSeance_ts,groupe_députés_affiliation,groupe&gvt_affiliation,groupe_all_affiliation,Texte_clean,repu_match_valide,FDM_Valeurs,FDM_Figures
0,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.323,2025-09-26,2018-06-02 09:30:00,LR,LR,LR,L'article 25 concerne les organismes HLM et le...,False,True,False
1,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.923,2025-09-26,2018-06-02 09:30:00,DEM,DEM,DEM,"Pour cette intervention sur l'article, je remp...",True,True,False
2,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.299,2025-09-26,2018-06-02 09:30:00,LR,LR,LR,C'est une réalité !,False,False,False
3,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.290,2025-09-26,2018-06-02 09:30:00,NI,NI,NI,C'est vrai !,False,False,False
4,CRSANR5L15S2018O1N245,NaN,NaN,20180602093000000,samedi 02 juin 2018,1,245,AN,15,Session ordinaire 2017-2018,...,0.923,2025-09-26,2018-06-02 09:30:00,DEM,DEM,DEM,"Nous devons tous en être conscients alors que,...",False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683655,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,NaN,2025-09-26,2023-02-27 16:00:00,NaN,GVT,REN,Il n'est pas un jour où je ne parle pas des se...,False,True,False
683656,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,0.000,2025-09-26,2023-02-27 16:00:00,GDR,GDR,GDR,"Monsieur le ministre, il faudrait des heures, ...",False,True,False
683657,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,NaN,2025-09-26,2023-02-27 16:00:00,NaN,GVT,REN,"Premièrement, est-il légitime pour l'État de s...",False,True,False
683658,CRSANR5L16S2023O1N156,RUANR5L16S2023IDS26837,SCR5A2023O1,20230227160000000,lundi 27 février 2023,1,156,AN,16,Session ordinaire 2022-2023,...,0.000,2025-09-26,2023-02-27 16:00:00,SOC-A,SOC-A,SOC-A,Je remercie le groupe GDR-NUPES d'avoir mis ce...,False,True,False


In [ ]:
pattern_dates = re.compile(
    r"(\b1789\b)"
    r"|(\b1790\b)"
    r"|(\b1791\b)"
    r"|(\b1792\b)"
    r"|(\b1793\b)"
    r"|(\b1794\b)"
    r"|(\b1795\b)"
    r"|(\b1799\b)"
    r"|(\b1802\b)"
    r"|(\b1804\b)"          
    r"|(\b1815\b)"
    r"|(\b1848\b)"              
    r"|(\b1852\b)"
    r"|(\b1870\b)"
    r"|(\b1875\b)"
    r"|(\b1879\b)"
    r"|(\b1881\b)"
    r"|(\b1882\b)"
    r"|(\b1894\b)"
    r"|(\b1899\b)"
    r"|(\b1901\b)"
    r"|(\b1905\b)"
    r"|(\b1906\b)"
    r"|(\bf[eé]vrier+1934\b)"
    r"|(\b1936\b)"
    r"|(\b1946\b)"
    r"|(\b1954\b)"
    r"|(\b1958\b)"
    r"|(\b1989\b)"
    r"|(\bR[eé]volution\b)"
    r"|(\bR[eé]sistance\b)"
    r"|(\b14+juillet\b)",
    re.I,
)

def dates(texte: str) -> bool:
    """Renvoie True si le texte contient une figure historique républicaine."""
    return bool(pattern_dates.search(texte))


In [40]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_dates"] = df["texte"].apply(dates)

In [14]:
# Appliquer sur la colonne
df["texte"] = df["texte"].fillna("") # nécessaire de remplacer les 35 NaN restantes par des chaînes vides pour faire tourner la fonction re
df["FDM_Historique"] = df["texte"].apply(famille_de_mot_figures)

In [ ]:
df

In [ ]:
# Regex des expressions à exclure


# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    r"|(" + pattern_pays + ")"  # ajout des exclusions de pays 5602 occurrences 5540 (sans le problème des 2 types d'apostrophes)
    r"|(\bGouvernement de la République française\b)" # 47 occurrences mais Pose question de si on met ou pas car surtout mobilisé lors de discussion sur la signature d'accords commerciaux entre deux pays (=signifie le gouvernement français + que la république comme idée politique)
    r"|(\badministration générale de la République\b)" #fait référence au nom d'une commission "commission des lois constitutionnelles, de la législation et de l’administration générale de la République"
    r"|(\brépublique[s] soviétique[s]\b)", # 6 occurrences 
    re.I,
)


def contains_lexical_outside_excl(text):
    # Trouver les positions des expressions exclues
    excl_positions = []

    # Ajouter les exclusions sensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )

    # Ajouter les exclusions insensibles à la casse
    excl_positions.extend(
         [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
     )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurences de la famille du mot 
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False